In [1]:
import sys
print(sys.executable)

D:\Anaconda_file\envs\retailpulse\python.exe


In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/cleaned_transactions.csv")

df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])

print("Shape:", df.shape)
print("Date range:", df["InvoiceDate"].min(), "to", df["InvoiceDate"].max())
print("Customers:", df["Customer ID"].nunique())

Shape: (42750, 9)
Date range: 2023-01-01 08:00:00 to 2024-09-05 18:00:00
Customers: 1200


In [3]:
# Define the cutoff date for churn analysis
cutoff_date = pd.Timestamp("2024-06-07")

# Transactions before the cutoff are used to create customer features
historical = df[df["InvoiceDate"] < cutoff_date].copy()

# Transactions after the cutoff are used to determine future churn
future = df[df["InvoiceDate"] >= cutoff_date].copy()

print("Historical transactions:", len(historical))
print("Future transactions:", len(future))
print("Historical date range:", historical["InvoiceDate"].min(), "to", historical["InvoiceDate"].max())
print("Future date range:", future["InvoiceDate"].min(), "to", future["InvoiceDate"].max())


Historical transactions: 42551
Future transactions: 199
Historical date range: 2023-01-01 08:00:00 to 2024-06-06 19:00:00
Future date range: 2024-06-07 08:00:00 to 2024-09-05 18:00:00


In [4]:
# Create customer-level features from historical transactions

customer_features = historical.groupby("Customer ID").agg(
    LastPurchase=("InvoiceDate", "max"),
    Frequency=("Invoice", "nunique"),
    Monetary=("Revenue", "sum"),
    TotalQuantity=("Quantity", "sum"),
    UniqueProducts=("StockCode", "nunique")
).reset_index()

# Recency in days from the cutoff date
customer_features["Recency"] = (
    cutoff_date - customer_features["LastPurchase"]
).dt.days

# Average order value
customer_features["AvgOrderValue"] = (
    customer_features["Monetary"] / customer_features["Frequency"]
)

print("Customer feature shape:", customer_features.shape)
print("\nFeatures:")
print(customer_features.head())

Customer feature shape: (1199, 8)

Features:
   Customer ID        LastPurchase  Frequency  Monetary  TotalQuantity  \
0      10000.0 2024-04-01 13:00:00         69   5658.65            632   
1      10001.0 2024-01-25 11:00:00         45   2675.70            357   
2      10002.0 2023-08-06 18:00:00         42   1700.28            213   
3      10003.0 2023-11-01 13:00:00         45   2157.64            281   
4      10004.0 2023-09-23 08:00:00         40   2421.04            283   

   UniqueProducts  Recency  AvgOrderValue  
0              63       66      82.009420  
1              41      133      59.460000  
2              40      305      40.482857  
3              44      218      47.947556  
4              35      257      60.526000  


In [5]:
# Identify customers who made a purchase after the cutoff date

future_customers = set(future["Customer ID"].unique())

customer_features["Churn"] = (
    ~customer_features["Customer ID"].isin(future_customers)
).astype(int)

print("Churn distribution:")
print(customer_features["Churn"].value_counts())

print("\nChurn percentage:")
print(customer_features["Churn"].value_counts(normalize=True).mul(100).round(2))

Churn distribution:
Churn
1    1194
0       5
Name: count, dtype: int64

Churn percentage:
Churn
1    99.58
0     0.42
Name: proportion, dtype: float64


In [6]:
# Check customer activity for different cutoff dates

for cutoff in [
    "2024-01-01",
    "2024-02-01",
    "2024-03-01",
    "2024-04-01",
    "2024-05-01"
]:
    cutoff = pd.Timestamp(cutoff)

    historical_count = df[df["InvoiceDate"] < cutoff]["Customer ID"].nunique()
    future_count = df[df["InvoiceDate"] >= cutoff]["Customer ID"].nunique()

    print(
        cutoff.date(),
        "| Historical customers:", historical_count,
        "| Future customers:", future_count
    )

2024-01-01 | Historical customers: 1185 | Future customers: 113
2024-02-01 | Historical customers: 1193 | Future customers: 63
2024-03-01 | Historical customers: 1195 | Future customers: 35
2024-04-01 | Historical customers: 1196 | Future customers: 22
2024-05-01 | Historical customers: 1199 | Future customers: 13


In [7]:
# Use a better cutoff date for churn prediction
cutoff_date = pd.Timestamp("2024-01-01")

# Historical and future transactions
historical = df[df["InvoiceDate"] < cutoff_date].copy()
future = df[df["InvoiceDate"] >= cutoff_date].copy()

# Create customer-level historical features
customer_features = historical.groupby("Customer ID").agg(
    LastPurchase=("InvoiceDate", "max"),
    Frequency=("Invoice", "nunique"),
    Monetary=("Revenue", "sum"),
    TotalQuantity=("Quantity", "sum"),
    UniqueProducts=("StockCode", "nunique")
).reset_index()

# Calculate recency
customer_features["Recency"] = (
    cutoff_date - customer_features["LastPurchase"]
).dt.days

# Calculate average order value
customer_features["AvgOrderValue"] = (
    customer_features["Monetary"] / customer_features["Frequency"]
)

# Identify customers who purchased during the future period
future_customers = set(future["Customer ID"].unique())

# Churn = 1 if customer made no future purchase
customer_features["Churn"] = (
    ~customer_features["Customer ID"].isin(future_customers)
).astype(int)

print("Customer feature shape:", customer_features.shape)
print("\nChurn distribution:")
print(customer_features["Churn"].value_counts())

print("\nChurn percentage:")
print(
    customer_features["Churn"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

Customer feature shape: (1185, 9)

Churn distribution:
Churn
1    1087
0      98
Name: count, dtype: int64

Churn percentage:
Churn
1    91.73
0     8.27
Name: proportion, dtype: float64


In [8]:
from sklearn.model_selection import train_test_split

# Select model features
feature_columns = [
    "Recency",
    "Frequency",
    "Monetary",
    "TotalQuantity",
    "UniqueProducts",
    "AvgOrderValue"
]

X = customer_features[feature_columns]
y = customer_features["Churn"]

# Stratified train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training samples:", X_train.shape[0])
print("Testing samples:", X_test.shape[0])

print("\nTraining churn distribution:")
print(y_train.value_counts())

print("\nTesting churn distribution:")
print(y_test.value_counts())

Training samples: 948
Testing samples: 237

Training churn distribution:
Churn
1    870
0     78
Name: count, dtype: int64

Testing churn distribution:
Churn
1    217
0     20
Name: count, dtype: int64


In [9]:
from sklearn.ensemble import RandomForestClassifier

# Create the churn prediction model
churn_model = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    class_weight="balanced",
    max_depth=8,
    min_samples_leaf=3,
    n_jobs=-1
)

# Train the model
churn_model.fit(X_train, y_train)

print("Churn model training completed.")

Churn model training completed.


In [10]:
from sklearn.metrics import (
    roc_auc_score,
    accuracy_score,
    classification_report,
    confusion_matrix
)

# Predictions
y_pred = churn_model.predict(X_test)
y_prob = churn_model.predict_proba(X_test)[:, 1]

# Evaluation metrics
auc_score = roc_auc_score(y_test, y_prob)
accuracy = accuracy_score(y_test, y_pred)

print(f"ROC-AUC: {auc_score:.4f}")
print(f"Accuracy: {accuracy:.4f}")

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

ROC-AUC: 1.0000
Accuracy: 1.0000

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        20
           1       1.00      1.00      1.00       217

    accuracy                           1.00       237
   macro avg       1.00      1.00      1.00       237
weighted avg       1.00      1.00      1.00       237


Confusion Matrix:
[[ 20   0]
 [  0 217]]


In [11]:
# Feature importance from the Random Forest

feature_importance = pd.DataFrame({
    "Feature": feature_columns,
    "Importance": churn_model.feature_importances_
}).sort_values("Importance", ascending=False)

print(feature_importance)

          Feature  Importance
0         Recency    0.520812
5   AvgOrderValue    0.232196
3   TotalQuantity    0.105173
2        Monetary    0.098810
1       Frequency    0.027631
4  UniqueProducts    0.015378


In [12]:
import shap

# Create SHAP explainer for the Random Forest
explainer = shap.TreeExplainer(churn_model)

# Calculate SHAP values for the test data
shap_values = explainer.shap_values(X_test)

print("SHAP values calculated successfully.")
print("Number of SHAP outputs:", len(shap_values))

D:\Anaconda_file\envs\retailpulse\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


SHAP values calculated successfully.
Number of SHAP outputs: 237


In [13]:
print("SHAP type:", type(shap_values))
print("SHAP shape:", np.array(shap_values).shape)


SHAP type: <class 'numpy.ndarray'>
SHAP shape: (237, 6, 2)


In [14]:
# Extract SHAP values for the churn class (Class 1)

shap_churn = shap_values[:, :, 1]

print("Churn SHAP shape:", shap_churn.shape)
print("Features:", feature_columns)

Churn SHAP shape: (237, 6)
Features: ['Recency', 'Frequency', 'Monetary', 'TotalQuantity', 'UniqueProducts', 'AvgOrderValue']


In [15]:
# SHAP summary plot for churn prediction

shap.summary_plot(
    shap_churn,
    X_test,
    feature_names=feature_columns,
    show=True
)

<Figure size 800x390 with 2 Axes>

In [16]:
import os
import joblib

# Make sure the models directory exists
os.makedirs("../models", exist_ok=True)

# Save the trained churn model
model_path = "../models/churn_model.pkl"
joblib.dump(churn_model, model_path)

print("Model saved successfully:")
print(model_path)

Model saved successfully:
../models/churn_model.pkl


In [17]:
# Save customer churn features and target

churn_data_path = "../data/churn_features.csv"

customer_features.to_csv(
    churn_data_path,
    index=False
)

print("Churn feature dataset saved successfully:")
print(churn_data_path)

print("\nShape:", customer_features.shape)


Churn feature dataset saved successfully:
../data/churn_features.csv

Shape: (1185, 9)


In [18]:
import joblib

loaded_model = joblib.load("../models/churn_model.pkl")

print("Model loaded successfully.")
print("Model type:", type(loaded_model).__name__)
print("Number of trees:", loaded_model.n_estimators)

Model loaded successfully.
Model type: RandomForestClassifier
Number of trees: 300
